# Challenge 5 — Merge / Join

**Nombre:** Miriam Ivonne Castañeda  
**Clase:** Merge / Join

## Objetivo

Integrar información de diferentes tablas utilizando claves y tipos de `join`.

En este ejercicio se utilizan:

- `dataset_limpio.csv`
- `regiones(1).csv`

También se crea una tabla sencilla de productos desde el dataset limpio.


## 1. Cargar los datos

In [2]:
import pandas as pd

df = pd.read_csv("/Users/ivonnecastaneda/Downloads/dataset_limpio.csv")
regiones = pd.read_csv("/Users/ivonnecastaneda/Downloads/regiones.csv")

print("Ventas:")
display(df.head())

print("Regiones:")
display(regiones.head())


Ventas:


,fecha,region_id,vendedor,producto_id,cantidad,nombre_producto,categoria,precio_unitario,total
0,2023-01-01,1,Vendedor_4,7,9,Producto 7,Electrónica,1932.73,17394.57
1,2023-01-01,3,Vendedor_10,5,5,Producto 5,Ropa,1240.04,6200.20
2,2023-01-01,5,Vendedor_6,1,1,Producto 1,Electrónica,1264.57,1264.57
3,2023-01-01,4,Vendedor_9,14,3,Producto 14,Deportes,2949.45,8848.35
4,2023-01-01,2,Vendedor_6,17,4,Producto 17,Ropa,1593.14,6372.56


Regiones:


,region_id,region,pais
0,1,Norte,México
1,2,Sur,México
2,3,Centro,México
3,4,Bajío,México
4,5,Sureste,México


## 2. Revisar las llaves

In [3]:
print("Columnas de ventas:")
print(df.columns.tolist())

print("\nColumnas de regiones:")
print(regiones.columns.tolist())

print("\nRegiones disponibles:")
display(regiones)


Columnas de ventas:
['fecha', 'region_id', 'vendedor', 'producto_id', 'cantidad', 'nombre_producto', 'categoria', 'precio_unitario', 'total']

Columnas de regiones:
['region_id', 'region', 'pais']

Regiones disponibles:


,region_id,region,pais
0,1,Norte,México
1,2,Sur,México
2,3,Centro,México
3,4,Bajío,México
4,5,Sureste,México


La llave que relaciona las dos tablas es **`region_id`**.

En `dataset_limpio.csv`, cada venta tiene un `region_id`.

En `regiones(1).csv`, cada `region_id` tiene información de la región y del país.


## 3. Revisar posibles duplicados en la llave

In [4]:
print("Duplicados de region_id en regiones:")
print(regiones["region_id"].duplicated().sum())

print("\nRegiones únicas:", regiones["region_id"].nunique())
print("Ventas:", len(df))


Duplicados de region_id en regiones:
0

Regiones únicas: 5
Ventas: 6000


## 4. Inner Join

In [5]:
inner = df.merge(regiones,on="region_id",how="inner")

print("Filas antes:", len(df))
print("Filas después del inner join:", len(inner))

display(inner.head())


Filas antes: 6000
Filas después del inner join: 6000


,fecha,region_id,vendedor,producto_id,cantidad,nombre_producto,categoria,precio_unitario,total,region,pais
0,2023-01-01,1,Vendedor_4,7,9,Producto 7,Electrónica,1932.73,17394.57,Norte,México
1,2023-01-01,3,Vendedor_10,5,5,Producto 5,Ropa,1240.04,6200.20,Centro,México
2,2023-01-01,5,Vendedor_6,1,1,Producto 1,Electrónica,1264.57,1264.57,Sureste,México
3,2023-01-01,4,Vendedor_9,14,3,Producto 14,Deportes,2949.45,8848.35,Bajío,México
4,2023-01-01,2,Vendedor_6,17,4,Producto 17,Ropa,1593.14,6372.56,Sur,México


El `inner join` conserva las ventas que tienen una región correspondiente.

Como todas las regiones del dataset tienen correspondencia, el número de filas se mantiene.


## 5. Left Join

In [6]:
left = df.merge(regiones,on="region_id",how="left")

print("Filas después del left join:", len(left))

print("\nFaltantes generados:")
print(left[["region", "pais"]].isna().sum())

Filas después del left join: 6000

Faltantes generados:
region    0
pais      0
dtype: int64


El `left join` mantiene todas las ventas del dataset principal.

En este caso tampoco aparecen faltantes en `region` o `pais`, porque todos los `region_id` tienen correspondencia.


## 6. Crear tabla de productos

In [8]:
productos = df[["producto_id", "nombre_producto", "categoria"]].drop_duplicates("producto_id")

print("Productos:", len(productos))
display(productos.head())

Productos: 20


,producto_id,nombre_producto,categoria
0,7,Producto 7,Electrónica
1,5,Producto 5,Ropa
2,1,Producto 1,Electrónica
3,14,Producto 14,Deportes
4,17,Producto 17,Ropa


## 7. Crear la tabla analítica final

In [9]:
tabla_final = left.merge(productos,on="producto_id",how="left",suffixes=("", "_producto"))

print("Filas:", len(tabla_final))
print("Columnas:", len(tabla_final.columns))

display(tabla_final.head())

Filas: 6000
Columnas: 13


,fecha,region_id,vendedor,producto_id,cantidad,nombre_producto,categoria,precio_unitario,total,region,pais,nombre_producto_producto,categoria_producto
0,2023-01-01,1,Vendedor_4,7,9,Producto 7,Electrónica,1932.73,17394.57,Norte,México,Producto 7,Electrónica
1,2023-01-01,3,Vendedor_10,5,5,Producto 5,Ropa,1240.04,6200.20,Centro,México,Producto 5,Ropa
2,2023-01-01,5,Vendedor_6,1,1,Producto 1,Electrónica,1264.57,1264.57,Sureste,México,Producto 1,Electrónica
3,2023-01-01,4,Vendedor_9,14,3,Producto 14,Deportes,2949.45,8848.35,Bajío,México,Producto 14,Deportes
4,2023-01-01,2,Vendedor_6,17,4,Producto 17,Ropa,1593.14,6372.56,Sur,México,Producto 17,Ropa


## 8. Validación

In [10]:
print("Filas originales:", len(df))
print("Filas finales:", len(tabla_final))

print("\nRegiones únicas:", tabla_final["region_id"].nunique())
print("Productos únicos:", tabla_final["producto_id"].nunique())

print("\nFaltantes en las columnas nuevas:")
print(tabla_final[["region", "pais", "nombre_producto", "categoria"]].isna().sum())


Filas originales: 6000
Filas finales: 6000

Regiones únicas: 5
Productos únicos: 20

Faltantes en las columnas nuevas:
region             0
pais               0
nombre_producto    0
categoria          0
dtype: int64


## 9. Preguntas que podemos responder

In [11]:
# 1. ¿Qué región genera más ventas?
ventas_region = (tabla_final.groupby(["region_id", "region"])["total"].sum().sort_values(ascending=False))
print("Ventas por región:")
display(ventas_region)

# 2. ¿Qué producto genera más ventas?
ventas_producto = (tabla_final.groupby(["producto_id", "nombre_producto"])["total"].sum().sort_values(ascending=False))
print("\nVentas por producto:")
display(ventas_producto.head())

# 3. ¿Qué categoría genera más ventas?
ventas_categoria = (tabla_final.groupby("categoria")["total"].sum().sort_values(ascending=False))

print("\nVentas por categoría:")
display(ventas_categoria)

Ventas por región:


region_id  region 
2          Sur        12850555.26
3          Centro     12737707.39
1          Norte      12272701.72
5          Sureste    12096084.01
4          Bajío      12027016.20
Name: total, dtype: float64


Ventas por producto:


producto_id  nombre_producto
13           Producto 13        5657119.42
14           Producto 14        5234997.57
4            Producto 4         5115906.98
9            Producto 9         4962168.70
15           Producto 15        4654430.73
Name: total, dtype: float64


Ventas por categoría:


categoria
Deportes       25661252.05
Ropa           18547804.74
Electrónica     8818502.58
Hogar           4962168.70
Alimentos       3994336.51
Name: total, dtype: float64

## 10. Conclusiones

1. `region_id` permitió relacionar correctamente las ventas con `regiones.csv`.
2. El `inner join` y el `left join` conservaron las 6,000 ventas porque todas tienen una región correspondiente.
3. La tabla final agrega el nombre de la región y el país a cada venta.
4. También se agregó información de productos usando `producto_id`.
5. Con la tabla final se pueden analizar las ventas por región, producto y categoría.
